In [1]:
# seq2seq_chatbot.py
import random
import math
import time
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

In [4]:

# Hyperparams (start small)
EMBED_SIZE = 256
HIDDEN_SIZE = 512
ENC_NUM_LAYERS = 1
DEC_NUM_LAYERS = 1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 20
TEACHER_FORCING_RATIO = 0.5
CLIP = 1.0  # gradient clipping
MAX_LEN = 50  # max seq len

In [5]:

toy_pairs = [
    ("hi", "hello"),
    ("how are you", "i am fine"),
    ("what is your name", "i am a chatbot"),
    ("tell me a joke", "why did the chicken cross the road"),
    ("bye", "goodbye"),
    ("thank you", "you are welcome"),
    ("how's the weather", "i do not know the weather"),
    ("i am hungry", "have you eaten?"),
    ("what is ai", "artificial intelligence"),
] * 200  # replicate to have enough examples


In [6]:

# ---------------------------
# Tokenization & Vocabulary
# ---------------------------
class Vocab:
    def __init__(self, min_freq=1):
        self.word2idx = {}
        self.idx2word = {}
        self.freqs = {}
        self.min_freq = min_freq
        # reserve indices
        for i, tok in enumerate([PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]):
            self.word2idx[tok] = i
            self.idx2word[i] = tok
        self.next_index = len(self.word2idx)

    def add_sentence(self, sent: str):
        for w in sent.strip().split():
            self.freqs[w] = self.freqs.get(w, 0) + 1

    def build(self):
        for w, f in sorted(self.freqs.items(), key=lambda x: -x[1]):
            if f >= self.min_freq and w not in self.word2idx:
                self.word2idx[w] = self.next_index
                self.idx2word[self.next_index] = w
                self.next_index += 1

    def numericalize(self, sent: str) -> List[int]:
        res = []
        for w in sent.strip().split():
            res.append(self.word2idx.get(w, self.word2idx[UNK_TOKEN]))
        return res

    def decode(self, indices: List[int]) -> str:
        words = []
        for idx in indices:
            if idx == self.word2idx[EOS_TOKEN]:
                break
            words.append(self.idx2word.get(idx, UNK_TOKEN))
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)


In [7]:

# Build vocab
vocab = Vocab(min_freq=1)
for s, t in toy_pairs:
    vocab.add_sentence(s)
    vocab.add_sentence(t)
vocab.build()


In [8]:

# ---------------------------
# Dataset & collate
# ---------------------------
class ChatDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]], vocab: Vocab, max_len=MAX_LEN):
        self.pairs = pairs
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src, tgt = self.pairs[idx]
        src_idx = [vocab.word2idx[SOS_TOKEN]] + vocab.numericalize(src) + [vocab.word2idx[EOS_TOKEN]]
        tgt_idx = [vocab.word2idx[SOS_TOKEN]] + vocab.numericalize(tgt) + [vocab.word2idx[EOS_TOKEN]]
        src_idx = src_idx[:self.max_len]
        tgt_idx = tgt_idx[:self.max_len]
        return torch.tensor(src_idx, dtype=torch.long), torch.tensor(tgt_idx, dtype=torch.long)

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_lens = [len(s) for s in src_batch]
    tgt_lens = [len(t) for t in tgt_batch]
    max_src = max(src_lens)
    max_tgt = max(tgt_lens)
    padded_src = torch.full((len(batch), max_src), vocab.word2idx[PAD_TOKEN], dtype=torch.long)
    padded_tgt = torch.full((len(batch), max_tgt), vocab.word2idx[PAD_TOKEN], dtype=torch.long)
    for i, (s, t) in enumerate(zip(src_batch, tgt_batch)):
        padded_src[i, :s.size(0)] = s
        padded_tgt[i, :t.size(0)] = t
    # lengths for pack_padded_sequence if desired
    return padded_src, src_lens, padded_tgt, tgt_lens

# Create DataLoader
dataset = ChatDataset(toy_pairs, vocab)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)


In [9]:

# ---------------------------
# Models: Encoder, Attention, Decoder
# ---------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=vocab.word2idx[PAD_TOKEN])
        self.gru = nn.GRU(embed_size, hidden_size, num_layers=n_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, hidden_size)  # to combine bidirectional
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths=None):
        # src: (batch, seq_len)
        embedded = self.dropout(self.embedding(src))  # (batch, seq_len, embed)
        # Optionally use pack_padded_sequence if you handle lengths
        outputs, hidden = self.gru(embedded)  # outputs: (batch, seq_len, hidden*2)
        # Combine forward + backward hidden (num_layers*2, batch, hidden)
        # We'll create a single-layer hidden for decoder
        # hidden: (n_layers*2, batch, hidden); combine directions
        # take last layer only if multiple layers used; here n_layers=1
        # combine forward/backward
        # hidden view: (n_layers, num_directions, batch, hidden)
        n_layers_times_dirs, batch_size, h = hidden.size()
        # reshape and combine
        hidden = hidden.view(1, 2, batch_size, h)  # (n_layers, num_dirs, batch, h)
        hidden = torch.cat((hidden[:,0,:,:], hidden[:,1,:,:]), dim=2)  # (n_layers, batch, h*2) but we want (1, batch, h*2)
        # pass through fc to collapse to hidden_size
        hidden = hidden.transpose(0,1)  # (batch, n_layers, h*2)
        hidden = hidden.contiguous().view(batch_size, -1)  # (batch, h*2)
        hidden = torch.tanh(self.fc(hidden))  # (batch, hidden)
        hidden = hidden.unsqueeze(0)  # (1, batch, hidden)
        return outputs, hidden  # outputs for attention, hidden for decoder init

class BahdanauAttention(nn.Module):
    def __init__(self, enc_hidden_size, dec_hidden_size, attn_dim):
        super().__init__()
        self.W1 = nn.Linear(enc_hidden_size * 2, attn_dim)  # encoder is bidirectional
        self.W2 = nn.Linear(dec_hidden_size, attn_dim)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, enc_outputs, dec_hidden):
        # enc_outputs: (batch, seq_len, enc_hidden*2)
        # dec_hidden: (1, batch, dec_hidden) -> make it (batch, dec_hidden)
        dec_hidden = dec_hidden.squeeze(0)  # (batch, dec_hidden)
        # Expand dec hidden to seq_len
        # Score = v(tanh(W1*enc_outputs + W2*dec_hidden_expanded))
        scores = self.v(torch.tanh(self.W1(enc_outputs) + self.W2(dec_hidden).unsqueeze(1))).squeeze(-1)  # (batch, seq_len)
        attn_weights = torch.softmax(scores, dim=1)  # (batch, seq_len)
        # Context vector
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)  # (batch, enc_hidden*2)
        return context, attn_weights  # weights for visualization

class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_size, enc_hidden_size, dec_hidden_size, attn_dim, n_layers=1, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=vocab.word2idx[PAD_TOKEN])
        self.attention = BahdanauAttention(enc_hidden_size, dec_hidden_size, attn_dim)
        self.gru = nn.GRU(embed_size + enc_hidden_size*2, dec_hidden_size, num_layers=n_layers, batch_first=True)
        self.fc_out = nn.Linear(dec_hidden_size + enc_hidden_size*2 + embed_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_step, last_hidden, enc_outputs):
        # input_step: (batch, ) token indices
        embedded = self.dropout(self.embedding(input_step)).unsqueeze(1)  # (batch,1,embed)
        context, attn_weights = self.attention(enc_outputs, last_hidden)  # context (batch, enc_h*2)
        # Combine embedded with context
        rnn_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)  # (batch,1,embed+enc_h*2)
        output, hidden = self.gru(rnn_input, last_hidden)  # output: (batch,1,dec_hidden)
        output = output.squeeze(1)  # (batch, dec_hidden)
        # concat output, context, embedded to predict
        pred_input = torch.cat([output, context, embedded.squeeze(1)], dim=1)
        output_vocab = self.fc_out(pred_input)  # (batch, vocab)
        return output_vocab, hidden, attn_weights


In [10]:

# ---------------------------
# Seq2Seq wrapper + helpers
# ---------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device=DEVICE):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, src_lens, tgt=None, teacher_forcing_ratio=0.5, max_len=MAX_LEN):
        # src: (batch, src_len)
        batch_size = src.size(0)
        vocab_size = len(vocab)
        # encode
        enc_outputs, enc_hidden = self.encoder(src, src_lens)
        # prepare first input tokens = SOS
        input_tok = torch.full((batch_size,), vocab.word2idx[SOS_TOKEN], dtype=torch.long, device=self.device)
        outputs = torch.zeros(batch_size, max_len, vocab_size, device=self.device)
        hidden = enc_hidden  # (1, batch, hidden)
        attn_weights_all = []
        for t in range(max_len):
            output_vocab, hidden, attn_weights = self.decoder(input_tok, hidden, enc_outputs)
            outputs[:, t, :] = output_vocab
            attn_weights_all.append(attn_weights.detach().cpu())
            # decide next input
            teacher_force = False
            if tgt is not None:
                teacher_force = random.random() < teacher_forcing_ratio
            top1 = output_vocab.argmax(1)
            input_tok = tgt[:, t] if (teacher_force and t < tgt.size(1)) else top1
        return outputs, attn_weights_all


In [11]:

# ---------------------------
# Training utilities
# ---------------------------
def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.xavier_uniform_(param.data)
        elif 'bias' in name:
            nn.init.constant_(param.data, 0)

def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    m = int(elapsed / 60)
    s = int(elapsed - (m * 60))
    return m, s


In [12]:

# ---------------------------
# Instantiate model, optimizer, criterion
# ---------------------------
enc = Encoder(len(vocab), EMBED_SIZE, HIDDEN_SIZE, n_layers=ENC_NUM_LAYERS).to(DEVICE)
dec = DecoderWithAttention(len(vocab), EMBED_SIZE, HIDDEN_SIZE, HIDDEN_SIZE, attn_dim=256, n_layers=DEC_NUM_LAYERS).to(DEVICE)
model = Seq2Seq(enc, dec, device=DEVICE).to(DEVICE)
model.apply(init_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.word2idx[PAD_TOKEN])


In [14]:

# ---------------------------
# Training loop (simple)
# ---------------------------
def train(model, dataloader, optimizer, criterion, clip=CLIP, teacher_forcing_ratio=TEACHER_FORCING_RATIO):
    model.train()
    epoch_loss = 0
    for src, src_lens, tgt, tgt_lens in dataloader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)
        optimizer.zero_grad()
        max_t = tgt.size(1)
        outputs, _ = model(src, src_lens, tgt=tgt, teacher_forcing_ratio=teacher_forcing_ratio, max_len=max_t)
        # outputs: (batch, max_t, vocab)
        # reshape for loss
        outputs = outputs.view(-1, outputs.size(2))
        tgt_flat = tgt.view(-1)
        loss = criterion(outputs, tgt_flat)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader)

def evaluate(model, dataloader, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, src_lens, tgt, tgt_lens in dataloader:
            src = src.to(DEVICE)
            tgt = tgt.to(DEVICE)
            max_t = tgt.size(1)
            outputs, _ = model(src, src_lens, tgt=None, teacher_forcing_ratio=0.0, max_len=max_t)
            outputs = outputs.view(-1, outputs.size(2))
            tgt_flat = tgt.view(-1)
            loss = criterion(outputs, tgt_flat)
            epoch_loss += loss.item()
    return epoch_loss / len(dataloader)


In [15]:

# ---------------------------
# Greedy decode single sentence
# ---------------------------
def respond(model, sentence: str, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src_idx = [vocab.word2idx[SOS_TOKEN]] + vocab.numericalize(sentence) + [vocab.word2idx[EOS_TOKEN]]
        src = torch.tensor(src_idx, dtype=torch.long).unsqueeze(0).to(DEVICE)
        src_lens = [len(src_idx)]
        enc_outputs, enc_hidden = model.encoder(src, src_lens)
        input_tok = torch.tensor([vocab.word2idx[SOS_TOKEN]], dtype=torch.long, device=DEVICE)
        hidden = enc_hidden
        tokens = []
        attns = []
        for _ in range(max_len):
            output_vocab, hidden, attn = model.decoder(input_tok, hidden, enc_outputs)
            top1 = output_vocab.argmax(1)
            tok = top1.item()
            if tok == vocab.word2idx[EOS_TOKEN]:
                break
            tokens.append(tok)
            attns.append(attn.cpu().numpy())
            input_tok = top1
        return vocab.decode(tokens), attns


In [16]:

# ---------------------------
# Run training
# ---------------------------
best_valid_loss = float('inf')
for epoch in range(1, MAX_EPOCHS + 1):
    start_time = time.time()
    train_loss = train(model, dataloader, optimizer, criterion)
    valid_loss = evaluate(model, dataloader, criterion)
    end_time = time.time()
    epoch_m, epoch_s = epoch_time(start_time, end_time)
    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {valid_loss:.4f} | Time: {epoch_m}m {epoch_s}s")
    # checkpoint
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), "seq2seq_best.pt")


Epoch 1 | Train Loss: 1.3615 | Val Loss: 0.1598 | Time: 0m 4s
Epoch 2 | Train Loss: 0.0546 | Val Loss: 0.0065 | Time: 0m 1s
Epoch 3 | Train Loss: 0.0037 | Val Loss: 0.0019 | Time: 0m 1s
Epoch 4 | Train Loss: 0.0016 | Val Loss: 0.0013 | Time: 0m 1s
Epoch 5 | Train Loss: 0.0012 | Val Loss: 0.0010 | Time: 0m 1s
Epoch 6 | Train Loss: 0.0009 | Val Loss: 0.0008 | Time: 0m 1s
Epoch 7 | Train Loss: 0.0007 | Val Loss: 0.0006 | Time: 0m 2s
Epoch 8 | Train Loss: 0.0006 | Val Loss: 0.0005 | Time: 0m 2s
Epoch 9 | Train Loss: 0.0005 | Val Loss: 0.0004 | Time: 0m 1s
Epoch 10 | Train Loss: 0.0355 | Val Loss: 0.5710 | Time: 0m 2s
Epoch 11 | Train Loss: 0.0510 | Val Loss: 0.0020 | Time: 0m 2s
Epoch 12 | Train Loss: 0.0012 | Val Loss: 0.0007 | Time: 0m 2s
Epoch 13 | Train Loss: 0.0006 | Val Loss: 0.0005 | Time: 0m 1s
Epoch 14 | Train Loss: 0.0004 | Val Loss: 0.0004 | Time: 0m 1s
Epoch 15 | Train Loss: 0.0003 | Val Loss: 0.0008 | Time: 0m 2s
Epoch 16 | Train Loss: 0.0003 | Val Loss: 0.0003 | Time: 0m 1s
E

In [17]:

# ---------------------------
# Test simple responses
# ---------------------------
examples = [
    "hi",
    "how are you",
    "what is your name",
    "i am hungry",
    "tell me a joke",
]

for ex in examples:
    reply, attn = respond(model, ex)
    print(f"> {ex} -> {reply}")


> hi -> <sos> i am fine
> how are you -> <sos> i am fine
> what is your name -> <sos> i am a chatbot
> i am hungry -> <sos> have you eaten?
> tell me a joke -> <sos> why did the chicken cross the road
